In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

In [57]:
vehicles = pd.read_csv("US_Accidents_March23_sampled_500k.csv", index_col=0)
df = vehicles.copy()

In [58]:
df

,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
ID,,,,,,,,,,,,,,,,,,,,,
A-2047758,Source2,2,2019-06-12 10:10:56,2019-06-12 10:55:58,30.641211,-91.153481,NaN,NaN,0.000,Accident on LA-19 Baker-Zachary Hwy at Lower Z...,...,False,False,False,False,True,False,Day,Day,Day,Day
A-4694324,Source1,2,2022-12-03 23:37:14.000000000,2022-12-04 01:56:53.000000000,38.990562,-77.399070,38.990037,-77.398282,0.056,Incident on FOREST RIDGE DR near PEPPERIDGE PL...,...,False,False,False,False,False,False,Night,Night,Night,Night
A-5006183,Source1,2,2022-08-20 13:13:00.000000000,2022-08-20 15:22:45.000000000,34.661189,-120.492822,34.661189,-120.492442,0.022,Accident on W Central Ave from Floradale Ave t...,...,False,False,False,False,True,False,Day,Day,Day,Day
A-4237356,Source1,2,2022-02-21 17:43:04,2022-02-21 19:43:23,43.680592,-92.993317,43.680574,-92.972223,1.054,Incident on I-90 EB near REST AREA Drive with ...,...,False,False,False,False,False,False,Day,Day,Day,Day
A-6690583,Source1,2,2020-12-04 01:46:00,2020-12-04 04:13:09,35.395484,-118.985176,35.395476,-118.985995,0.046,RP ADV THEY LOCATED SUSP VEH OF 20002 - 726 CR...,...,False,False,False,False,False,False,Night,Night,Night,Night
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
A-6077227,Source1,2,2021-12-15 07:30:00,2021-12-15 07:50:30,45.522510,-123.084104,45.520225,-123.084211,0.158,Stationary traffic on OR-47 from NW Martin Rd ...,...,False,False,False,False,False,False,Night,Day,Day,Day
A-6323243,Source1,2,2021-12-19 16:25:00,2021-12-19 17:40:37,26.702570,-80.111169,26.703141,-80.111133,0.040,Incident on MILITARY TRL near WESTGATE AVE Dri...,...,False,False,False,False,False,False,Day,Day,Day,Day
A-3789256,Source1,2,2022-04-13 19:28:29,2022-04-13 21:33:44,34.561862,-112.259620,34.566822,-112.267150,0.549,Crash on the right shoulder on E SR-69 Northbo...,...,False,False,False,False,True,False,Night,Night,Day,Day


In [59]:
df.isnull().sum()

Source                        0
Severity                      0
Start_Time                    0
End_Time                      0
Start_Lat                     0
Start_Lng                     0
End_Lat                  220377
End_Lng                  220377
Distance(mi)                  0
Description                   1
Street                      691
City                         19
County                        0
State                         0
Zipcode                     116
Country                       0
Timezone                    507
Airport_Code               1446
Weather_Timestamp          7674
Temperature(F)            10466
Wind_Chill(F)            129017
Humidity(%)               11130
Pressure(in)               8928
Visibility(mi)            11291
Wind_Direction            11197
Wind_Speed(mph)           36987
Precipitation(in)        142616
Weather_Condition         11101
Amenity                       0
Bump                          0
Crossing                      0
Give_Way

In [60]:
# So to start with lets identify the obvious columns that we do not want to work with since it is irrelevant for our purposes of classifying the severity of accidents
df = df.drop(columns=[
    'Source', # Where the data is sourced from
    # We dont want any of the coordinates we have the country, county and street, state
    'Start_Lat', 
    'Start_Lng', 
    'End_Lat', 
    'End_Lng', 
    'Country', # 100% of the values in Country is the US and we know that this dataset is from the US so we can take it out 
    'Airport_Code', # No need for this one
    'Timezone', # We do not need a timezone such as US/Cental or US/Pacific since we want to know what time of day is relevant at each place
    'Description' # We dont need the human language description it adds no value since it just states that an accident happened in a specific county
])

In [61]:
# Next we have 7 numerical weather condition columns, we want to check if any are redundant so we will start by checking correlation between them
# Temperature, Wind_Chill, Humidity, Pressure, Visibility, Wind_Speed, Precipitation
corr = df[['Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)',
           'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 
           'Precipitation(in)']].corr()

print(corr)
# From the correlation matrix we see that Wind_Chill is very strongly correlated with Temperature p = 0.993741 we also know that about 1/4 of that column is missing so we will drop it
df = df.drop(columns=['Wind_Chill(F)'])

                   Temperature(F)  Wind_Chill(F)  Humidity(%)  Pressure(in)  \
Temperature(F)           1.000000       0.993741    -0.330652      0.108418   
Wind_Chill(F)            0.993741       1.000000    -0.314226      0.085624   
Humidity(%)             -0.330652      -0.314226     1.000000      0.118809   
Pressure(in)             0.108418       0.085624     0.118809      1.000000   
Visibility(mi)           0.213359       0.229615    -0.380681      0.035473   
Wind_Speed(mph)          0.032820      -0.044444    -0.171128     -0.023768   
Precipitation(in)       -0.004382      -0.011547     0.083448      0.017199   

                   Visibility(mi)  Wind_Speed(mph)  Precipitation(in)  
Temperature(F)           0.213359         0.032820          -0.004382  
Wind_Chill(F)            0.229615        -0.044444          -0.011547  
Humidity(%)             -0.380681        -0.171128           0.083448  
Pressure(in)             0.035473        -0.023768           0.017199  
Visibil

In [62]:
severity_counts = df['Severity'].value_counts().sort_index()
severity_percent = df['Severity'].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({
    'Count': severity_counts,
    'Percentage (%)': severity_percent.round(2)
})

print(summary)
# We can see that the data is heavily skewed with 80% of accidents being at severity 2

           Count  Percentage (%)
Severity                        
1           4274            0.85
2         398142           79.63
3          84520           16.90
4          13064            2.61


In [63]:
# We also have 4 different types of telling whether its day or night we want to check if we can drop all of them or all but one
# Sunrise_Sunset, Civil_Twilight, Nautical_Twilight, Astronomical_Twilight, They all have either Day or Night so well convert them to 0 and 1 and check correlations

for col in ['Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']:
    df[col] = df[col].map({'Day': 1, 'Night': 0})

corr = df[['Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']].corr()
print(corr)
# These as expected are all very correlated but keeping them all would be redundant since they all measure the same "lightness" level
# There are two options we could take here, the simpler one being just keeping the most intuative one, Sunrise_Sunset and dropping the rest
# The other one being some combination of at least the twilight columns creating a sort of Light_Score or Darkness column where we would convert all of them to binary encodings and add them together

# So for now we will drop the twiliht columns, since sunrise sunset is a hard cutoff of day and night and the rest is like a spectrum with astronomical twilight being the most strict.
df = df.drop(columns=['Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight'])

                       Sunrise_Sunset  Civil_Twilight  Nautical_Twilight  \
Sunrise_Sunset               1.000000        0.891315           0.777282   
Civil_Twilight               0.891315        1.000000           0.872013   
Nautical_Twilight            0.777282        0.872013           1.000000   
Astronomical_Twilight        0.681685        0.764833           0.877018   

                       Astronomical_Twilight  
Sunrise_Sunset                      0.681685  
Civil_Twilight                      0.764833  
Nautical_Twilight                   0.877018  
Astronomical_Twilight               1.000000  


In [64]:
# Next we have 13 True/False boolean features, viewing them at first glance we see that some of them are mostly False or atleast > 99.5% False
# But before we drop any of them prematurely we want to see if they are skewed towards severity, for example most of the True ones in the columns where 99.9% are False could be those that have severity 4
bool_cols = df.select_dtypes(include='bool').columns

for col in bool_cols:
    print(pd.crosstab(df[col], df['Severity'], normalize='index'))

# After checking the frequency of the True or False values with severity we can see that Turning_Loop is 100% False so we will just drop that column,
# Roundabout also has only True values in the most common class (severity 2) so its not tellig us anything so we will drop it.
df = df.drop(columns=['Roundabout', 'Turning_Loop'])

Severity         1         2         3         4
Amenity                                         
False     0.008479  0.794732  0.170600  0.026188
True      0.014053  0.920045  0.044581  0.021321
Severity         1         2         3         4
Bump                                            
False     0.008548  0.796254  0.169071  0.026127
True      0.009479  0.867299  0.094787  0.028436
Severity         1         2         3         4
Crossing                                        
False     0.006993  0.782115  0.183291  0.027601
True      0.020817  0.908058  0.056619  0.014506
Severity         1         2         3         4
Give_Way                                        
False     0.008508  0.796193  0.169211  0.026087
True      0.016935  0.815411  0.132938  0.034716
Severity         1         2         3         4
Junction                                        
False     0.008782  0.802086  0.163776  0.025357
True      0.005621  0.723701  0.234902  0.035777
Severity         1  

In [65]:
missing = df.isnull().sum().sort_values(ascending=False)

missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_%': (missing / len(df)) * 100,
    'Dtype': df.dtypes
})

missing_df = missing_df[missing_df['Missing_Count'] > 0]
#print(missing)
print(missing_df)

                   Missing_Count  Missing_%    Dtype
City                          19     0.0038   object
Humidity(%)                11130     2.2260  float64
Precipitation(in)         142616    28.5232  float64
Pressure(in)                8928     1.7856  float64
Street                       691     0.1382   object
Sunrise_Sunset              1483     0.2966  float64
Temperature(F)             10466     2.0932  float64
Visibility(mi)             11291     2.2582  float64
Weather_Condition          11101     2.2202   object
Weather_Timestamp           7674     1.5348   object
Wind_Direction             11197     2.2394   object
Wind_Speed(mph)            36987     7.3974  float64
Zipcode                      116     0.0232   object


In [66]:
# We have 5 numerical columns only ony has a missing value percentage of above 5% that is Wind_Speed(mph) so for the rest we will just fill the rest (4) with the column median
df['Temperature(F)'] = df['Temperature(F)'].fillna(df['Temperature(F)'].median())
df['Humidity(%)'] = df['Humidity(%)'].fillna(df['Humidity(%)'].median())
df['Pressure(in)'] = df['Pressure(in)'].fillna(df['Pressure(in)'].median())
df['Visibility(mi)'] = df['Visibility(mi)'].fillna(df['Visibility(mi)'].median())
# The categorical columns all have a very low percentage of missing values so well just fill them with unknown
df['City'] = df['City'].fillna("Unknown")
df['Street'] = df['Street'].fillna("Unknown")
df['Zipcode'] = df['Zipcode'].fillna("Unknown")
df['Weather_Condition'] = df['Weather_Condition'].fillna("Unknown")
df['Wind_Direction'] = df['Wind_Direction'].fillna("Unknown")
df['Sunrise_Sunset'] = df['Sunrise_Sunset'].fillna("Unknown")
df['Weather_Timestamp'] = df['Weather_Timestamp'].fillna("Unknown")

In [76]:
# For the more difficult columns Precipitation(in) with 28.5% missing values and Wind_Speed(mph) with 7.4% missing values
missing_precip_by_sev = df.groupby('Severity')['Precipitation(in)'].apply(
    lambda x: x.isnull().mean() * 100
)
print(missing_precip_by_sev)
print(df['Precipitation(in)'].value_counts())
# So for the precipitation we see that most of the values that are not missing is 0 which means no rain
# We do see that the missigness is not evenly spread across severity so we should probably add some missing indicator for this column and then fill it with 0
df['Precipitation_missing'] = df['Precipitation(in)'].isnull().astype(int) # 1 if value was missing from precipitation 0 otherwise
df['Precipitation(in)'] = df['Precipitation(in)'].fillna(0)
# And since Wind Speed also has a pretty large amount of missing values we will also create a msising indicator for it but instead of filling with 0 we will fill it with the median
df['Wind_Speed_missing'] = df['Wind_Speed(mph)'].isnull().astype(int)
df['Wind_Speed(mph)'] = df['Wind_Speed(mph)'].fillna(df['Wind_Speed(mph)'].median())

Severity
1    0.0
2    0.0
3    0.0
4    0.0
Name: Precipitation(in), dtype: float64
Precipitation(in)
0.00    465564
0.01      9725
0.02      4735
0.03      3217
0.04      2405
         ...  
2.31         1
1.15         1
1.89         1
4.89         1
1.39         1
Name: count, Length: 175, dtype: int64


In [79]:
# Final check if we still have any values that are missing
missing = df.isnull().sum().sort_values(ascending=False)
missing

Severity                 0
Start_Time               0
End_Time                 0
Distance(mi)             0
Street                   0
City                     0
County                   0
State                    0
Zipcode                  0
Weather_Timestamp        0
Temperature(F)           0
Humidity(%)              0
Pressure(in)             0
Visibility(mi)           0
Wind_Direction           0
Wind_Speed(mph)          0
Precipitation(in)        0
Weather_Condition        0
Amenity                  0
Bump                     0
Crossing                 0
Give_Way                 0
Junction                 0
No_Exit                  0
Railway                  0
Station                  0
Stop                     0
Traffic_Calming          0
Traffic_Signal           0
Sunrise_Sunset           0
Precipitation_missing    0
Wind_Speed_missing       0
dtype: int64

In [ ]:
# We need to turn all the boolean (True/False) columns into 1/0 for the model
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
df